In [1]:
!pip install -q transformers sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.0 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
documents = [
    "RAG stands for Retrieval Augmented Generation. It retrieves relevant information before generating an answer.",
    "Self-RAG allows a language model to retrieve information and critique its own generated response.",
    "GraphRAG represents relationships between information using a graph structure.",
    "Corrective RAG evaluates retrieved documents and can perform additional retrieval when the retrieved information is insufficient.",
    "ReAct combines reasoning and actions such as searching or retrieving information.",
    "Large Language Models can hallucinate by generating information that is unsupported or incorrect.",
    "Sentence Transformers can convert text into numerical embeddings for semantic similarity and retrieval."
]

In [5]:
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Documents indexed:", index.ntotal)

Documents indexed: 7


In [6]:
def retrieve(query, k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        results.append({
            "text": documents[idx],
            "distance": float(distance)
        })

    return results

In [7]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
def generate_answer(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=True
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [9]:
def self_rag(query):

    # STEP 1: Retrieve evidence
    retrieved = retrieve(query, k=3)

    context = "\n".join(
        [f"- {r['text']}" for r in retrieved]
    )

    # STEP 2: Initial generation
    prompt = f"""
You are a factual AI assistant.

Use the following retrieved evidence to answer the question.

Evidence:
{context}

Question:
{query}

Answer only using information supported by the evidence.
"""

    answer = generate_answer(prompt)

    # STEP 3: Self-critique
    critique_prompt = f"""
Evaluate the following answer against the provided evidence.

Evidence:
{context}

Question:
{query}

Answer:
{answer}

Determine:
1. Is the answer fully supported?
2. Is more evidence required?
3. Is there any unsupported or contradictory information?

Return:
SUPPORTED
or
NEEDS_RETRIEVAL

Then briefly explain why.
"""

    critique = generate_answer(critique_prompt)

    # STEP 4: Decide whether another retrieval is required
    needs_retrieval = "NEEDS_RETRIEVAL" in critique.upper()

    if needs_retrieval:

        # Additional retrieval
        retrieved_more = retrieve(query, k=5)

        context_more = "\n".join(
            [f"- {r['text']}" for r in retrieved_more]
        )

        # STEP 5: Regenerate using expanded evidence
        revision_prompt = f"""
Answer the question using the evidence below.

Question:
{query}

Evidence:
{context_more}

Previous answer:
{answer}

Critique:
{critique}

Generate a corrected answer.
Do not invent facts that are not supported by the evidence.
"""

        final_answer = generate_answer(revision_prompt)

        return {
            "query": query,
            "initial_answer": answer,
            "critique": critique,
            "additional_retrieval": True,
            "final_answer": final_answer,
            "evidence": retrieved_more
        }

    return {
        "query": query,
        "initial_answer": answer,
        "critique": critique,
        "additional_retrieval": False,
        "final_answer": answer,
        "evidence": retrieved
    }

In [10]:
result = self_rag(
    "What is Self-RAG and how does it help with hallucinations?"
)

print("QUERY:")
print(result["query"])

print("\nINITIAL ANSWER:")
print(result["initial_answer"])

print("\nSELF-CRITIQUE:")
print(result["critique"])

print("\nADDITIONAL RETRIEVAL:")
print(result["additional_retrieval"])

print("\nFINAL ANSWER:")
print(result["final_answer"])

QUERY:
What is Self-RAG and how does it help with hallucinations?

INITIAL ANSWER:
Self-RAG, or Self-Revised Augmented Retrieval, is a technique that allows a language model to revise its responses based on feedback from previous iterations of itself. This process helps in reducing hallucinations by ensuring that the model's output is more aligned with the context provided during training. The self-revision aspect ensures that the model learns to generate coherent and accurate responses over time, thereby minimizing errors and improving overall performance. Additionally, Self-RAG supports the use of Retrieval-Augmented Generation (RAG), which enhances the accuracy of the model's answers by retrieving relevant information from external sources before generating a response. When combined, these techniques provide a robust framework for refining language models' outputs, making them less likely to produce unrealistic or incorrect results.

SELF-CRITIQUE:
To evaluate whether the given answ